# Multimodalni model: Sentinel-2 + otisci zgrada (zajednički trening)

Snimak i otisci nose komplementaran signal (tekstura naselja sa satelita vs prebrojiva
izgrađenost iz otisaka), pa umesto da ih treniramo odvojeno i spajamo predikcije, ovde
jedan model dobija oba ulaza odjednom i sam uči kako da ih iskombinuje. To je fuzija koju
smo dogovorili sa profesorom: jedan model koji istovremeno dobija strukturirane podatke i
sliku. Tri grane:

| grana | ulaz | trup |
|---|---|---|
| satelitska | Sentinel-2 isečak, 6 opsega | ResNet-18 (ImageNet), 512-dim |
| rasterska | otisci zgrada, 2 kanala | ResNet-18 (ImageNet), 512-dim |
| tabelarna | 14 strukturiranih atributa otisaka (iz geometrije) | MLP, 32-dim |

Spajamo sve tri reprezentacije (1056-dim) i puštamo ih u zajedničku regresionu glavu.
Predviđa se `log1p(broj_stanovnika)`, trening end-to-end (glava pa fine-tuning oba trupa).
Ista GroupKFold podela po opštinama i isti OOF protokol kao u ostalim notebucima, pa su
rezultati direktno uporedivi (i sa stacking fuzijom).

## Instalacija

In [ ]:
%pip install -q timm "mlflow>=3.0"

## Konfiguracija

In [ ]:
import os
import sys
# koren repoa na sys.path (penji se od radnog dir dok ne nadje core/)
koren = os.getcwd()
while not os.path.isdir(os.path.join(koren, "core")) and os.path.dirname(koren) != koren:
    koren = os.path.dirname(koren)
sys.path.insert(0, koren)

import glob

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import mlflow
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

from core import (
    seed_everything, two_phase_train,
    channel_stats, NW, seed_worker,
    make_folds, run_metrics, summary_line,
    cv_summary_figure,
    setup_mlflow, output_dir, save_oof,
)
from scripts import config     # paths do podataka i spisak strukturiranih atributa

OUT_DIR = output_dir()   # tezine modela i OOF parquet (UC Volume, lokalno "out/")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CFG = {
    "sat_bands": 6,                 # Sentinel-2 opsega (satelitska grana)
    "fp_bands": 2,                  # kanali otisaka: pokrivenost, gustina zgrada
    "tab_hidden": 32,               # izlaz tabelarne grane (MLP nad strukturiranim atributima)
    "tab_dropout": 0.15,
    "img_px": 224,
    "embed_hidden": 256,            # skriveni sloj zajednicke glave (1024 -> 256 -> 1)
    "dropout": 0.2,
    "epochs_head": 3,
    "epochs_finetune": 40,
    "batch_size": 48,               # dva ResNet-18 trupa -> nesto manji batch nego kod jednog
    "head_lr": 1e-3,
    "finetune_lr": 3e-4,
    "seed": 42,
}
seed_everything(CFG["seed"])
print("device:", DEVICE,
      "| sat cutouts:", len(os.listdir(config.CUTOUTS)),
      "| footprint cutouts:", len(os.listdir(config.FOOTPRINT_CUT)))

## Podaci i podela (presek naselja sa oba ulaza)

In [ ]:
labele = pd.read_parquet(config.NASELJE_TABLE)[
    ["naselje_maticni_broj", "opstina_maticni_broj", "pop"]]

def paths(folder):
    t = pd.DataFrame({"path": glob.glob(os.path.join(folder, "*.npy"))})
    t["naselje_maticni_broj"] = t.path.map(lambda f: int(os.path.splitext(os.path.basename(f))[0]))
    return t

sat = paths(config.CUTOUTS).rename(columns={"path": "path_sat"})
fp  = paths(config.FOOTPRINT_CUT).rename(columns={"path": "path_fp"})
df = (
    sat.merge(fp, on="naselje_maticni_broj", how="inner")   # samo naselja sa OBA ulaza
    .merge(labele, on="naselje_maticni_broj", how="inner")
)
df["y"] = np.log1p(df["pop"]).astype("float32")

# tabelarna grana: isti strukturirani atributi kao u 03 (iz scripts/config.py),
# svi iz geometrije; FP_LOG idu kroz log1p pre standardizacije.
LOG_ATRIBUTI = config.FP_LOG
ATRIBUTI = config.FP_ATRIBUTI

atr = pd.read_parquet(config.NASELJE_FOOTPRINTS)[["naselje_maticni_broj", *ATRIBUTI]]
for c in LOG_ATRIBUTI:
    atr[c] = np.log1p(atr[c].clip(lower=0))
df = df.merge(atr, on="naselje_maticni_broj", how="inner")   # sva tri modaliteta

print(f"sat {len(sat)} | footprint {len(fp)} | presek (sva tri ulaza + labela) {len(df)}")

FOLDS = make_folds(df)
N_FOLDS = len(FOLDS)
broj_opstina = df["opstina_maticni_broj"].nunique()
print(f"uzoraka {len(df)} | opstina {broj_opstina} | foldova {N_FOLDS}")
for i, (t, v) in enumerate(FOLDS):
    print(f"  fold {i}: trening {len(t)} / val {len(v)} naselja ({v['opstina_maticni_broj'].nunique()} opstina)")

## Normalizacija i dataset

In [ ]:
class MultimodalDataset(Dataset):
    # Satelitski isecak + footprint raster + atributi istog naselja, uz log1p(pop). Ista
    # augmentacija ide na oba rastera jer su prostorno poravnati.

    def __init__(self, frame, mean_s, std_s, mean_f, std_f, skaler, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.mean_s, self.std_s = mean_s, std_s
        self.mean_f, self.std_f = mean_f, std_f
        self.tab = skaler.transform(self.frame[ATRIBUTI].values).astype("float32")
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        red = self.frame.iloc[i]
        xs = (np.load(red.path_sat).astype("float32") - self.mean_s[0]) / self.std_s[0]
        xf = (np.load(red.path_fp).astype("float32") - self.mean_f[0]) / self.std_f[0]
        if self.augment:
            if np.random.rand() < 0.5:
                xs = xs[:, :, ::-1]
                xf = xf[:, :, ::-1]
            if np.random.rand() < 0.5:
                xs = xs[:, ::-1, :]
                xf = xf[:, ::-1, :]
            okreta = np.random.randint(4)
            xs = np.rot90(xs, okreta, axes=(1, 2))
            xf = np.rot90(xf, okreta, axes=(1, 2))
        return (
            torch.from_numpy(np.ascontiguousarray(xs)),
            torch.from_numpy(np.ascontiguousarray(xf)),
            torch.from_numpy(self.tab[i]),
            torch.tensor([red.y], dtype=torch.float32),
        )


def make_loaders_mm(train_frame, val_frame):
    # Vrati (train_dl, val_dl); normalizacija po modalitetu iz trening skupa ovog folda.
    mean_s, std_s = channel_stats(train_frame.path_sat.tolist())
    mean_f, std_f = channel_stats(train_frame.path_fp.tolist())
    # atributi: standardizacija fitovana samo na trening foldu (isto kao channel_stats)
    skaler = StandardScaler().fit(train_frame[ATRIBUTI].values)
    gen = torch.Generator().manual_seed(CFG["seed"])
    _sw = (lambda wid: seed_worker(wid, CFG["seed"])) if NW else None

    tdl = DataLoader(MultimodalDataset(train_frame, mean_s, std_s, mean_f, std_f, skaler, augment=True),
                     batch_size=CFG["batch_size"], shuffle=True,
                     # batch od tacno 1 uzorka rusi BatchNorm u trening modu
                     drop_last=len(train_frame) % CFG["batch_size"] == 1,
                     generator=gen, worker_init_fn=_sw,
                     num_workers=NW, pin_memory=True, persistent_workers=NW > 0,
                     prefetch_factor=4 if NW else None)
    vdl = DataLoader(MultimodalDataset(val_frame, mean_s, std_s, mean_f, std_f, skaler),
                     batch_size=CFG["batch_size"],
                     num_workers=NW, pin_memory=True, persistent_workers=NW > 0,
                     prefetch_factor=4 if NW else None)
    return tdl, vdl

## Model (dve grane + zajednička glava)

In [ ]:
class MultimodalModel(nn.Module):
    # Dva ResNet-18 trupa + MLP nad atributima, konkatenacija u zajednicku glavu. Glava i
    # tabelarna grana imaju prefiks "head", pa ih two_phase_train (head_prefix="head") u
    # fazi 1 trenira same, uz zamrznute trupove.

    def __init__(self):
        super().__init__()
        self.sat = timm.create_model("resnet18", pretrained=True,
                                     in_chans=CFG["sat_bands"], num_classes=0)
        self.fp  = timm.create_model("resnet18", pretrained=True,
                                     in_chans=CFG["fp_bands"], num_classes=0)
        self.head_tab = nn.Sequential(
            nn.Linear(len(ATRIBUTI), CFG["tab_hidden"]),
            nn.BatchNorm1d(CFG["tab_hidden"]),
            nn.ReLU(),
            nn.Dropout(CFG["tab_dropout"]),
        )
        d = self.sat.num_features + self.fp.num_features + CFG["tab_hidden"]
        self.head = nn.Sequential(
            nn.Linear(d, CFG["embed_hidden"]),
            nn.ReLU(),
            nn.Dropout(CFG["dropout"]),
            nn.Linear(CFG["embed_hidden"], 1),
        )

    def forward(self, xs, xf, xt):
        return self.head(torch.cat([self.sat(xs), self.fp(xf), self.head_tab(xt)], dim=1))


loss_fn = nn.HuberLoss()
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def run_pass_mm(net, loader, treniraj, optim=None, freeze_bn=False):
    # Jedan prolaz; kao core.train.run_pass ali sa tri ulaza po uzorku.
    net.train(treniraj)
    if freeze_bn:                    # faza 1: trupovi zamrznuti -> ne azuriraj BN statistiku
        for modul in net.modules():
            if isinstance(modul, nn.BatchNorm2d):
                modul.eval()

    ukupno, P, Y = 0.0, [], []
    for xs, xf, xt, y in loader:
        xs = xs.to(DEVICE, non_blocking=True)
        xf = xf.to(DEVICE, non_blocking=True)
        xt = xt.to(DEVICE, non_blocking=True)
        y  = y.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(treniraj), torch.autocast("cuda", enabled=use_amp):
            out  = net(xs, xf, xt)
            loss = loss_fn(out, y)

        if treniraj:
            optim.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()

        ukupno += loss.item() * len(y)
        P.append(out.detach().float().cpu().numpy())
        Y.append(y.cpu().numpy())

    return ukupno / len(loader.dataset), np.concatenate(P).ravel(), np.concatenate(Y).ravel()

## Trening (sa MLflow praćenjem)

In [ ]:
def train_fold(train_frame, val_frame):
    # Jedan fold: glava pa fine-tuning; vraca (best_state, r2, oof_pred, net).
    train_dl, val_dl = make_loaders_mm(train_frame, val_frame)
    net = MultimodalModel().to(DEVICE)

    def epoch(opt, korak, freeze_bn=False):
        tl, _, _ = run_pass_mm(net, train_dl, True, opt, freeze_bn=freeze_bn)
        vl, P, Y = run_pass_mm(net, val_dl, False)
        r2 = r2_score(Y, P)
        mlflow.log_metrics({"train_loss": tl, "val_loss": vl, "val_r2": r2}, step=korak)
        return r2

    best_r2, best_state = two_phase_train(
        net, epoch,
        CFG["epochs_head"], CFG["epochs_finetune"],
        CFG["head_lr"],     CFG["finetune_lr"],
        head_prefix="head",
    )
    net.load_state_dict(best_state)
    _, P, _ = run_pass_mm(net, val_dl, False)
    oof_pred_pop = np.clip(np.expm1(P), 0, None)   # OOF predikcija u populaciji
    return best_state, best_r2, oof_pred_pop, net

## Evaluacija

In [ ]:
def run():
    # Puna k-struka GroupKFold CV (multimodalni pristup).
    setup_mlflow()   # Databricks workspace; lokalno mlruns/, ili Databricks preko env varijabli
    oof = pd.Series(np.nan, index=df.naselje_maticni_broj.values, dtype="float32")
    fold_r2 = []

    with mlflow.start_run(run_name=f"multimodal-cv{len(FOLDS)}"):
        mlflow.log_params(CFG)
        mlflow.log_params({
            "pristup": "multimodal",
            "backbone": "2x resnet18 (sat 6ch + footprint 2ch) + MLP nad atributima, concat 1056 -> head",
            "n_atributa": len(ATRIBUTI),
            "pretrained": True,
            "optimizer": "AdamW",
            "scheduler": "CosineAnnealingLR",
            "loss": "HuberLoss",
            "target": "log1p(pop)",
            "cv": f"GroupKFold(opstina) x{len(FOLDS)}",
            "n_uzoraka": len(df),
            "n_opstina": int(broj_opstina),
        })

        for fold, (train_frame, val_frame) in enumerate(FOLDS):
            with mlflow.start_run(run_name=f"multimodal-fold{fold}", nested=True):
                mlflow.log_params({**CFG, "fold": fold,
                                   "n_train": len(train_frame), "n_val": len(val_frame)})
                best_state, best_r2, oof_pred_pop, net = train_fold(train_frame, val_frame)
                mlflow.log_metric("best_val_r2", best_r2)
                oof.loc[val_frame.naselje_maticni_broj.values] = oof_pred_pop
                put = f"{OUT_DIR}/multimodal_fold{fold}.pt"
                torch.save(best_state, put)
                mlflow.log_artifact(put)
                mlflow.pytorch.log_model(
                    net,
                    name=f"model_multimodal_fold{fold}",
                    serialization_format="pickle"
                )
                fold_r2.append(best_r2)
                print(f"[fold {fold}] best val R2 {best_r2:.3f}")

        # === agregacija preko svih foldova (OOF: svako naselje predvidjeno tacno jednom) ===
        oof_pred = oof.loc[df.naselje_maticni_broj.values].values.astype("float32")
        stvarno  = df["pop"].values.astype("float32")
        agg = run_metrics(fold_r2, stvarno, oof_pred, df)
        mlflow.log_metrics(agg)
        put_oof = save_oof(df, oof_pred, "multimodal", OUT_DIR)   # i ulaz za stacking poredjenje
        mlflow.log_artifact(put_oof)
        fig = cv_summary_figure(fold_r2, agg, stvarno, oof_pred, df, label="multimodal")
        plt.show()
        mlflow.log_figure(fig, "cv_evaluacija_multimodal.png")

    print(summary_line(agg, "multimodal"))
    return {"pristup": "multimodal", **agg}


rezultat = run()
print("\n=== Rezultat (multimodalni model) ===")
display(pd.DataFrame([rezultat]).set_index("pristup"))